# Speculative Decoding from Scratch

Build order:
1. **Stage 1** — Baseline greedy decoding (no `model.generate()`)
2. **Stage 2** — Core speculative decoding loop
3. **Stage 3** — Metrics & correctness verification
4. **Stage 4** — Code edit workload adaptation

Model pair: `gpt2` (draft) + `gpt2-xl` (target)  
Small enough to run on CPU, big enough to show meaningful acceptance rate differences.

## Setup

In [ ]:
!pip install transformers torch --quiet

In [ ]:
import torch
import torch.nn.functional as F
import time
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import Optional
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
# Load models — gpt2 as draft, gpt2-xl as target
# ~500MB + ~6GB. If RAM-constrained, swap gpt2-xl -> gpt2-large (~3GB)

DRAFT_MODEL_NAME = "gpt2"
TARGET_MODEL_NAME = "gpt2-xl"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(TARGET_MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

print("Loading draft model...")
draft_model = AutoModelForCausalLM.from_pretrained(DRAFT_MODEL_NAME).to(device).eval()

print("Loading target model...")
target_model = AutoModelForCausalLM.from_pretrained(TARGET_MODEL_NAME).to(device).eval()

print(f"Draft params:  {sum(p.numel() for p in draft_model.parameters()):,}")
print(f"Target params: {sum(p.numel() for p in target_model.parameters()):,}")

---
## Stage 1 — Baseline Autoregressive Decoding

Implement greedy decoding manually. No `model.generate()`.  
Goal: get comfortable with logit shapes and KV cache plumbing before touching speculative decoding.

In [ ]:
def greedy_decode(
    model,
    input_ids: torch.Tensor,       # shape: [1, seq_len]
    max_new_tokens: int = 50,
) -> tuple[torch.Tensor, float]:
    """
    Greedy autoregressive decoding with KV cache.
    Returns (output_ids, tokens_per_second).

    Steps:
      1. Run model on full prompt to get initial KV cache
      2. At each step, pass only the last token + past_key_values
      3. Take argmax of logits[:, -1, :] to get next token
      4. Append to sequence, repeat

    Useful shapes to know:
      logits:          [batch, seq_len, vocab_size]
      logits[:, -1, :] [batch, vocab_size]  <- only last position matters
    """
    generated = input_ids.clone()
    past_key_values = None
    t0 = time.time()

    with torch.no_grad():
        for _ in range(max_new_tokens):
            # TODO: run model forward pass
            # Hint: pass `past_key_values` and `use_cache=True`
            # On first step, feed full `generated`
            # On subsequent steps, feed only the last token
            raise NotImplementedError

            # TODO: extract next token (greedy = argmax)
            # next_token shape should be [1, 1]
            raise NotImplementedError

            # TODO: append next_token to generated
            raise NotImplementedError

    elapsed = time.time() - t0
    tps = max_new_tokens / elapsed
    return generated, tps

In [ ]:
# --- Sanity check ---
prompt = "The capital of France is"
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

# Test on draft model first (faster iteration)
output_ids, tps = greedy_decode(draft_model, input_ids, max_new_tokens=20)
print(tokenizer.decode(output_ids[0]))
print(f"Draft model: {tps:.1f} tok/s")

output_ids, tps = greedy_decode(target_model, input_ids, max_new_tokens=20)
print(tokenizer.decode(output_ids[0]))
print(f"Target model: {tps:.1f} tok/s")

---
## Stage 2 — Speculative Decoding

Three components to implement independently, then wire together.

```
draft_autoregressive  →  generates k candidate tokens
target_score          →  scores all k tokens in one forward pass  
rejection_sample      →  accepts/rejects per token, samples correction
```

### 2a — Draft: generate k tokens autoregressively

In [ ]:
def draft_autoregressive(
    draft_model,
    input_ids: torch.Tensor,   # shape: [1, seq_len]
    k: int,
    temperature: float = 0.0,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Run draft model autoregressively to produce k candidate tokens.

    Args:
      temperature: 0.0 for greedy (argmax), > 0 for sampling.

    Returns:
      draft_tokens: [k]        integer token ids
      draft_probs:  [k, vocab] probability distribution at each step

    Key point: save the full probability distribution (softmax of logits),
    not just the argmax — you need p_draft(xᵢ) in the rejection step.
    """
    draft_tokens = []
    draft_probs = []
    past_key_values = None
    current_ids = input_ids.clone()

    with torch.no_grad():
        for _ in range(k):
            # TODO: forward pass through draft model (with KV cache)
            raise NotImplementedError

            # TODO: get logits at last position, then compute probs
            # If temperature > 0: probs = softmax(logits / temperature), then sample via multinomial
            # If temperature == 0: probs = softmax(logits), then pick argmax
            raise NotImplementedError

            # TODO: store next_token and probs, append to current_ids
            raise NotImplementedError

    draft_tokens = torch.stack(draft_tokens)          # [k]
    draft_probs = torch.stack(draft_probs)            # [k, vocab]
    return draft_tokens, draft_probs

In [ ]:
# Quick check
tokens, probs = draft_autoregressive(draft_model, input_ids, k=5)
print("Draft tokens:", tokenizer.decode(tokens.tolist()))
print("draft_tokens shape:", tokens.shape)    # expect [5]
print("draft_probs  shape:", probs.shape)     # expect [5, vocab_size]
print("Probs sum to 1?:", probs.sum(dim=-1))  # should all be ~1.0

### 2b — Target: score k draft tokens in one forward pass

> This is the trickiest part. Read the docstring carefully before implementing.

In [ ]:
def target_score(
    target_model,
    input_ids: torch.Tensor,      # shape: [1, prefix_len]  — context before draft
    draft_tokens: torch.Tensor,   # shape: [k]              — draft candidates
    temperature: float = 0.0,
) -> torch.Tensor:
    """
    Run target model ONCE over (context + draft_tokens), return
    the target's probability distributions at each draft position.

    Args:
      temperature: 0.0 for raw softmax (greedy), > 0 to apply
        softmax(logits / T) so target probs live on the same scale
        as the draft probs used in rejection sampling.

    Returns:
      target_probs: [k+1, vocab]  — +1 because we also get the bonus
                                    distribution at position k (for free!)

    Indexing intuition:
      Input sequence:  [t₀, t₁, ..., t_{n-1}, x₁, x₂, ..., xₖ]
      Logit positions:  0   1  ...   n-1       n  n+1  ...  n+k-1

      target_probs[i] = target's distribution GIVEN context ending at xᵢ
                      = softmax(logits[:, n-1+i, :])   for i = 1..k

      The (k+1)-th distribution (logits[:, n+k-1, :]) is a bonus token
      from the target — use it when all k drafts are accepted.

    Common mistake: off-by-one on which logit positions to slice.
    Draw out the sequence indices before coding.
    """
    # Build full input: context + draft tokens
    draft_ids = draft_tokens.unsqueeze(0)              # [1, k]
    full_input = torch.cat([input_ids, draft_ids], dim=-1)  # [1, prefix+k]

    with torch.no_grad():
        # TODO: single forward pass through target model (no KV cache needed here)
        raise NotImplementedError

    # TODO: slice out the k+1 relevant positions
    # Hint: you want logits at positions [prefix_len-1 : prefix_len+k]
    # Then apply softmax — if temperature > 0, divide logits by temperature first
    raise NotImplementedError

    # target_probs shape: [k+1, vocab]
    return target_probs

In [ ]:
# Quick check
target_probs = target_score(target_model, input_ids, tokens)
print("target_probs shape:", target_probs.shape)       # expect [6, vocab_size] (k+1)
print("Probs sum to 1?:", target_probs.sum(dim=-1))    # should all be ~1.0

# Sanity: what token does target predict at position 0?
target_choice = target_probs[0].argmax().item()
draft_choice = tokens[0].item()
print(f"\nPosition 0 — draft: '{tokenizer.decode([draft_choice])}' | target: '{tokenizer.decode([target_choice])}'")

### 2c — Rejection sampling

The core statistical guarantee. Implement the acceptance criterion and the correction distribution.

In [ ]:
def rejection_sample(
    draft_tokens: torch.Tensor,  # [k]
    draft_probs:  torch.Tensor,  # [k, vocab]
    target_probs: torch.Tensor,  # [k+1, vocab]  — k+1 from target_score
    greedy: bool = True,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Accept/reject each draft token, with two modes:

    greedy=True  (temperature=0):
      Accept xᵢ iff xᵢ == argmax(target_probs[i]).
      Correction / bonus is argmax of the target distribution.
      Guarantees identical output to greedy target decoding.

    greedy=False (temperature>0, true rejection sampling):
      Accept xᵢ with probability min(1, p_target(xᵢ) / p_draft(xᵢ)).
      On rejection, sample correction from normalize(max(0, p_target - p_draft)).
      Bonus token sampled from target_probs[k].
      Guarantees the output distribution is exactly p_target.

    Returns:
      accepted:   [n_accepted]  accepted draft tokens (0 <= n_accepted <= k)
      correction: [1]           one correction/bonus token
    """
    accepted = []

    for i in range(len(draft_tokens)):
        if greedy:
            # TODO: get target's greedy choice via argmax
            # Accept if draft_tokens[i] matches, otherwise return correction
            raise NotImplementedError
        else:
            # TODO: compute accept_prob = min(1, p_target(token) / p_draft(token))
            # Sample r ~ Uniform(0,1). If r < accept_prob, accept.
            # Otherwise compute residual = clamp(target_probs[i] - draft_probs[i], min=0)
            # and sample correction from normalized residual.
            raise NotImplementedError

    # All k tokens accepted — pick bonus token
    if greedy:
        # TODO: argmax of target_probs[k]
        raise NotImplementedError
    else:
        # TODO: sample from target_probs[k] via multinomial
        raise NotImplementedError

    return torch.stack(accepted), bonus_token

In [ ]:
# Quick check
accepted, correction = rejection_sample(tokens, probs, target_probs)
print(f"k=5, accepted={len(accepted)} tokens")
print(f"Accepted: '{tokenizer.decode(accepted.tolist())}'")
print(f"Correction token: '{tokenizer.decode(correction.tolist())}'")

### 2d — Full speculative decode loop

In [ ]:
@dataclass
class SpecDecodeStats:
    acceptance_rates: list = field(default_factory=list)  # per-step α
    tokens_per_pass:  list = field(default_factory=list)  # tokens produced per target pass
    n_target_passes:  int = 0
    total_tokens:     int = 0
    wall_time:        float = 0.0

    @property
    def mean_acceptance_rate(self):
        return np.mean(self.acceptance_rates) if self.acceptance_rates else 0.0

    @property
    def mean_tokens_per_pass(self):
        return np.mean(self.tokens_per_pass) if self.tokens_per_pass else 0.0

    @property
    def tokens_per_second(self):
        return self.total_tokens / self.wall_time if self.wall_time > 0 else 0.0

In [ ]:
def speculative_decode(
    draft_model,
    target_model,
    input_ids: torch.Tensor,
    k: int = 5,
    max_new_tokens: int = 50,
    temperature: float = 0.0,
) -> tuple[torch.Tensor, SpecDecodeStats]:
    """
    Full speculative decoding loop.

    Args:
      temperature: 0.0 for greedy decoding, > 0 for sampling.
        - greedy: draft uses argmax, rejection uses argmax comparison.
          Output is identical to greedy target decoding.
        - sampling: draft samples from softmax(logits/T), rejection uses
          stochastic accept/reject. Output distribution matches target.

    Each iteration:
      1. Draft generates k candidates
      2. Target scores all k in one pass
      3. Rejection sampling accepts n <= k tokens + 1 correction
      4. Append accepted + correction to generated sequence
      5. Repeat until max_new_tokens reached
    """
    greedy = (temperature == 0.0)
    generated = input_ids.clone()
    stats = SpecDecodeStats()
    t0 = time.time()
    initial_len = input_ids.shape[1]

    while (generated.shape[1] - initial_len) < max_new_tokens:
        # TODO: call draft_autoregressive (pass temperature)
        raise NotImplementedError

        # TODO: call target_score (pass temperature so probs match the draft scale)
        raise NotImplementedError

        # TODO: call rejection_sample (pass greedy flag)
        raise NotImplementedError

        # TODO: append accepted + correction to generated
        raise NotImplementedError

        # TODO: update stats
        # acceptance_rate this step = len(accepted) / k
        # tokens_per_pass this step = len(accepted) + 1
        raise NotImplementedError

    generated = generated[:, :initial_len + max_new_tokens]
    stats.wall_time = time.time() - t0
    stats.total_tokens = generated.shape[1] - initial_len
    return generated, stats

In [ ]:
# End-to-end test
prompt = "The capital of France is"
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

output, stats = speculative_decode(draft_model, target_model, input_ids, k=5, max_new_tokens=40)
print(tokenizer.decode(output[0]))
print(f"\nMean acceptance rate:  {stats.mean_acceptance_rate:.2%}")
print(f"Mean tokens per pass:  {stats.mean_tokens_per_pass:.2f}  (theoretical max: k+1={6})")
print(f"Target forward passes: {stats.n_target_passes}")
print(f"Tokens/sec:            {stats.tokens_per_second:.1f}")

---
## Stage 3 — Metrics & Correctness Verification

Two things to verify:
1. **Statistical correctness** — speculative outputs match target distribution
2. **Speedup** — measure actual wallclock improvement vs baseline

### 3a — Greedy correctness: outputs must be identical to target baseline

In [ ]:
def verify_greedy_correctness(
    draft_model,
    target_model,
    tokenizer,
    prompts: list[str],
    k: int = 5,
    max_new_tokens: int = 20,
) -> dict:
    """
    For each prompt, compare greedy output (temperature=0) from:
      (a) pure target model — ground truth
      (b) speculative decoding

    With greedy decoding both MUST be identical (deterministic).
    This is the easiest correctness check — if greedy outputs differ,
    you have a bug in indexing or the rejection step.

    TODO: implement this comparison
      - Encode each prompt and store the prefix_len
      - Call greedy_decode(target_model, ...) for the ground truth
      - Call speculative_decode(..., temperature=0.0) for speculative
      - Decode both and compare strings exactly
    """
    results = {}
    for prompt in prompts:
        input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
        prefix_len = input_ids.shape[1]

        # TODO: Ground truth — pure target model, greedy
        target_out, _ = greedy_decode(target_model, input_ids, max_new_tokens)
        target_text = tokenizer.decode(target_out[0][prefix_len:])

        # TODO: Speculative — must pass temperature=0.0
        spec_out, _ = speculative_decode(
            draft_model, target_model, input_ids, k, max_new_tokens, temperature=0.0,
        )
        spec_text = tokenizer.decode(spec_out[0][prefix_len:])

        results[prompt] = {
            "target": target_text,
            "speculative": spec_text,
            "match": target_text == spec_text,
        }
    return results

In [ ]:
test_prompts = [
    "The capital of France is",
    "In machine learning, gradient descent",
    "The transformer architecture was introduced",
    "Python is a programming language that",
]

results = verify_greedy_correctness(draft_model, target_model, tokenizer, test_prompts, k=5)

for prompt, r in results.items():
    status = "✓" if r["match"] else "✗  BUG"
    print(f"{status}  '{prompt[:45]}'")
    if not r["match"]:
        print(f"       target:      {r['target'][:80]}")
        print(f"       speculative: {r['speculative'][:80]}")

match_rate = sum(r["match"] for r in results.values()) / len(results)
print(f"\nMatch rate: {match_rate:.0%}  (must be 100% for greedy)")

### 3a-bis — Statistical verification for temperature sampling

With `temperature > 0`, speculative decoding uses stochastic rejection sampling.
Outputs are **no longer deterministic**, so we can't compare strings.
Instead we verify that the **token-level distribution** of speculative decoding
matches the target model's distribution by:

1. Computing the **analytical** target distribution via a single forward pass
   (`softmax(logits / T)`) — this is exact, no sampling needed.
2. Running `n_samples` speculative decoding generations and collecting
   the empirical distribution of the *first generated token*.
3. Computing the **Total Variation (TV) distance** between analytical and empirical.
4. TV ≈ 0 means the distributions match — the rejection sampler is correct.

In [ ]:
def analytical_target_distribution(
    target_model,
    input_ids: torch.Tensor,
    temperature: float = 1.0,
) -> torch.Tensor:
    """
    Compute the exact next-token distribution from the target model
    in a single forward pass. No sampling needed — this is the ground truth.

    TODO: implement
      - Run target_model(input_ids) to get logits
      - Return softmax(logits[:, -1, :] / temperature)[0]  — shape [vocab]
    """
    # TODO: implement this function
    pass


def verify_sampling_distribution(
    draft_model,
    target_model,
    tokenizer,
    prompt: str,
    temperature: float = 1.0,
    k: int = 5,
    n_samples: int = 500,
) -> dict:
    """
    Compare the empirical first-token distribution from speculative decoding
    against the ANALYTICAL target distribution (exact, computed in one forward pass).

    Using the analytical target eliminates sampling noise on the reference side,
    so the TV distance reflects only spec-decode variance and any algorithmic bias.

    TODO: implement statistical verification
      - Encode prompt, get prefix_len and vocab_size
      - Call analytical_target_distribution to get exact target_dist [vocab]
      - Create spec_counts (zero tensor of size vocab_size)
      - For each of n_samples:
        - Run speculative_decode(temperature=T, max_new_tokens=1)
        - Get first generated token → increment spec_counts
      - spec_dist = spec_counts / spec_counts.sum()
      - TV distance = 0.5 * sum(|target_dist - spec_dist|)
      - Return dict with tv_distance, target_dist, spec_dist, top_tokens
    """
    # TODO: implement this function
    pass

In [ ]:
test_prompts_sampling = [
    "The capital of France is",
    "In machine learning, gradient descent",
]
temperature = 1.0
n_samples = 500

print(f"Statistical verification  (T={temperature}, n={n_samples} spec samples, analytical target)\n")

for prompt in test_prompts_sampling:
    result = verify_sampling_distribution(
        draft_model, target_model, tokenizer,
        prompt, temperature=temperature, n_samples=n_samples,
    )
    tv = result["tv_distance"]
    status = "✓" if tv < 0.10 else "⚠"
    print(f"{status}  '{prompt[:45]}'  TV distance = {tv:.4f}")

    top = result["top_tokens"]
    labels = [tokenizer.decode([t]).strip() or f"[{t}]" for t in top]
    t_probs = [result["target_dist"][t].item() for t in top]
    s_probs = [result["spec_dist"][t].item() for t in top]

    x = np.arange(len(labels))
    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 3))
    ax.bar(x - width / 2, t_probs, width, label="target (analytical)", alpha=0.8)
    ax.bar(x + width / 2, s_probs, width, label="speculative (empirical)", alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_ylabel("Probability")
    ax.set_title(f"'{prompt[:40]}…'  —  TV = {tv:.4f}")
    ax.legend()
    plt.tight_layout()
    plt.show()

print("\nTV < 0.10 indicates distributions match (some sampling noise expected).")

### 3b — Speedup benchmark

In [ ]:
def benchmark(
    draft_model,
    target_model,
    tokenizer,
    prompt: str,
    k_values: list[int] = [1, 3, 5, 8, 10],
    max_new_tokens: int = 100,
    n_runs: int = 3,
) -> dict:
    """
    Measure wallclock time and tokens/sec for:
      - Baseline (target only)
      - Speculative decoding at each k

    Run n_runs times and take the median to reduce noise.
    """
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    results = {"baseline": [], "speculative": {}}

    # Baseline
    for _ in range(n_runs):
        _, tps = greedy_decode(target_model, input_ids, max_new_tokens)
        results["baseline"].append(tps)
    results["baseline_median"] = np.median(results["baseline"])

    # Speculative at each k
    for k in k_values:
        runs = []
        acceptance_rates = []
        for _ in range(n_runs):
            _, stats = speculative_decode(draft_model, target_model, input_ids, k, max_new_tokens)
            runs.append(stats.tokens_per_second)
            acceptance_rates.append(stats.mean_acceptance_rate)
        results["speculative"][k] = {
            "tps_median": np.median(runs),
            "speedup": np.median(runs) / results["baseline_median"],
            "mean_alpha": np.mean(acceptance_rates),
        }

    return results

In [ ]:
bench_prompt = "In the field of artificial intelligence, large language models have"
bench_results = benchmark(draft_model, target_model, tokenizer, bench_prompt)

print(f"Baseline (target only): {bench_results['baseline_median']:.1f} tok/s\n")
print(f"{'k':>4}  {'tok/s':>8}  {'speedup':>8}  {'α (accept)':>12}")
print("-" * 40)
for k, v in bench_results["speculative"].items():
    print(f"{k:>4}  {v['tps_median']:>8.1f}  {v['speedup']:>8.2f}x  {v['mean_alpha']:>12.2%}")

In [ ]:
# Plot speedup vs k, colored by acceptance rate
k_vals = list(bench_results["speculative"].keys())
speedups = [bench_results["speculative"][k]["speedup"] for k in k_vals]
alphas = [bench_results["speculative"][k]["mean_alpha"] for k in k_vals]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(k_vals, speedups, "o-", color="steelblue", linewidth=2)
ax1.axhline(1.0, linestyle="--", color="gray", label="baseline")
ax1.set_xlabel("k (draft tokens)"); ax1.set_ylabel("Speedup vs baseline")
ax1.set_title("Speedup vs k"); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(k_vals, alphas, "o-", color="coral", linewidth=2)
ax2.set_xlabel("k (draft tokens)"); ax2.set_ylabel("Mean acceptance rate α")
ax2.set_title("Acceptance rate vs k"); ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 1)

plt.tight_layout()
plt.show()

---
## Stage 4 — Code Edit Workload

The reason speculative decoding shines on code edits: edit outputs contain
large stretches of **unchanged lines** (the context lines in a diff). These
are trivially predicted by the draft model → α close to 1.0 for those spans.

We'll measure how acceptance rate changes between open-ended generation vs edit-style prompts.

In [ ]:
# Edit-style prompts vs open-ended
# These simulate the kind of output structure code editors generate

edit_prompts = [
    # Lots of verbatim repetition in the output = high α
    """Complete this diff:
-    def forward(self, x):
-        return self.linear(x)
+    def forward(self, x: torch.Tensor) -> torch.Tensor:
+""",
    """Continue this Python function by adding type hints:
def compute_loss(predictions, targets, weights=None):
    # Add type annotations to this function
    def compute_loss(""",
]

open_ended_prompts = [
    "Explain the intuition behind attention mechanisms in transformers:",
    "Write a short story about a robot learning to paint:",
]

def measure_alpha_by_domain(prompts, label, k=5):
    alphas = []
    for prompt in prompts:
        input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
        _, stats = speculative_decode(draft_model, target_model, input_ids, k=k, max_new_tokens=60)
        alphas.append(stats.mean_acceptance_rate)
    print(f"{label}: mean α = {np.mean(alphas):.2%}  (per prompt: {[f'{a:.0%}' for a in alphas]})")
    return alphas

edit_alphas = measure_alpha_by_domain(edit_prompts, "Edit-style ")
open_alphas = measure_alpha_by_domain(open_ended_prompts, "Open-ended")

In [ ]:
# Bar chart comparing acceptance rates by domain
fig, ax = plt.subplots(figsize=(7, 4))
categories = ["Edit-style", "Open-ended"]
means = [np.mean(edit_alphas), np.mean(open_alphas)]
colors = ["steelblue", "coral"]

bars = ax.bar(categories, means, color=colors, alpha=0.8, width=0.4)
ax.set_ylabel("Mean acceptance rate α")
ax.set_title("Acceptance rate: edit-style vs open-ended prompts")
ax.set_ylim(0, 1)
for bar, v in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.02, f"{v:.0%}",
            ha="center", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

# The key insight: edit-style prompts should have noticeably higher α
# because the draft model easily predicts verbatim/unchanged code lines

---
## Extension Exercises

Once stages 1–4 are working, these are natural next steps:

**1. ~~Temperature sampling~~ ✅ Implemented above**  
`draft_autoregressive`, `rejection_sample`, and `speculative_decode` all accept  
a `temperature` parameter. Greedy (T=0) and sampling (T>0) are both supported  
and statistically verified in Stage 3a-bis.

**2. Dynamic k**  
Instead of fixed k, adapt k per step based on recent acceptance rate.  
If α is high → increase k. If α is dropping → decrease k.

**3. Tree-based speculative decoding**  
Instead of a single draft sequence, generate a *tree* of k branches.  
Target scores all branches in one pass. Higher expected tokens per pass.

**4. Medusa heads**  
Add multiple draft heads directly to the target model (no separate draft model).  
Each head predicts token at offset +1, +2, ..., +k from current position.

**5. Swap in a code model pair**  
Try `codegen-350M` (draft) + `codegen-2B` (target) and repeat the acceptance rate
analysis on real code edit prompts. α should be much higher than with GPT-2.